In [0]:
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import *

VOLUME_PATH="/Volumes/olist_ecommerce/bronze/raw_files"
CATALOG="olist_ecommerce"

print("Setup complete!")
print(f"Reading from: {VOLUME_PATH}")

In [0]:
def ingest_To_Bronze (csv_name, table_name, batch_id="batch_001", mode="overwrite"):
    """
    Đọc CSV từ Volume → thêm metadata → write Delta table vào Unity Catalog.
    mode='overwrite' cho lần đầu, 'append' cho incremental load.
    """
    df = (spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{VOLUME_PATH}/{csv_name}.csv")
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_batch_id",    lit(batch_id))
        .withColumn("_source_file", lit(csv_name)))
    
    (df.write
        .format("delta")
        .mode(mode)
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.bronze.{table_name}"))

    count = spark.sql(f"SELECT COUNT(*) as cnt FROM {CATALOG}.bronze.{table_name}").collect()[0]["cnt"]
    print(f"[Bronze] {table_name}: {count:,} rows ({mode})")

In [0]:
tables = {
    "olist_orders_dataset":              "orders",
    "olist_order_payments_dataset":      "payments",
    "olist_order_reviews_dataset":       "reviews",
    "olist_order_items_dataset":         "order_items", 
    "olist_products_dataset":            "products",
    "olist_customers_dataset":           "customers",
    "olist_sellers_dataset":             "sellers",
    "olist_geolocation_dataset":         "geolocation",
    "product_category_name_translation": "category_map",
}

for csv_name, table_name in tables.items():
    ingest_To_Bronze(csv_name, table_name, batch_id="batch_001", mode="overwrite")

print("\nBatch 1 complete!")

In [0]:
spark.sql("""
    SELECT table_name, 
           row_count
    FROM (
        SELECT 'orders'      AS table_name, COUNT(*) AS row_count FROM olist_ecommerce.bronze.orders      UNION ALL
        SELECT 'payments',                  COUNT(*)              FROM olist_ecommerce.bronze.payments     UNION ALL
        SELECT 'reviews',                   COUNT(*)              FROM olist_ecommerce.bronze.reviews      UNION ALL
        SELECT 'products',                  COUNT(*)              FROM olist_ecommerce.bronze.products     UNION ALL
        SELECT 'customers',                 COUNT(*)              FROM olist_ecommerce.bronze.customers    UNION ALL
        SELECT 'sellers',                   COUNT(*)              FROM olist_ecommerce.bronze.sellers      UNION ALL
        SELECT 'geolocation',               COUNT(*)              FROM olist_ecommerce.bronze.geolocation  UNION ALL
        SELECT 'category_map',              COUNT(*)              FROM olist_ecommerce.bronze.category_map UNION ALL
        SELECT "order_items",               count(*)              FROM olist_ecommerce.bronze.order_items
    )
    ORDER BY row_count DESC
""").show()